# CourtVision — Phase 3: Court Calibration

Clean rebuild of the original Phase 3 notebook (archived in `notebooks/archive/`). Reusable logic lives in the
`courtvision/` package; this notebook only orchestrates and displays results.

| Phase | Status |
|---|---|
| 1 Detection | YOLO26n fine-tuned (`best.pt`) |
| 2 Tracking | Optimized ByteTrack (`match_thresh=0.90`): IDF1 47.89 %, MOTA 60.54 %, 76 ID switches |
| 3 Calibration | **In progress.** A single static homography is invalid (moving broadcast camera). |

**Conventions** (see `docs/phase3_calibration.md`)
- Frame numbers in calibration = `FRAME_PATHS` index (0-based). Tracking `frame` column = file stem. Convert with `data.frame_id_of / index_of`.
- Lane corners order: baseline-lower, baseline-upper, FT-upper, FT-lower. Court coordinates are **lane-relative metres** (NCAA lane, unverified for this court).
- Manual annotations live in `courtvision/annotations.py` (verbatim; frames 105/115 are the references).

## 1 · Setup

In [ ]:
%pip -q install "ultralytics>=8.4" "opencv-python-headless>=4.9" "pandas>=2.2" "PyYAML>=6" "trackers"

In [ ]:
# Make the `courtvision` package importable: repo checkout (local / git clone) or a Kaggle dataset upload.
import sys
from pathlib import Path

def find_repo():
    candidates = [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/Courtvision")]
    kin = Path("/kaggle/input")
    if kin.exists():
        for pattern in ("*/courtvision/__init__.py", "*/*/courtvision/__init__.py", "*/*/*/courtvision/__init__.py"):
            candidates += [p.parent.parent for p in kin.glob(pattern)]
    for c in candidates:
        if (c / "courtvision" / "__init__.py").exists():
            return c
    raise FileNotFoundError("courtvision package not found. Attach the repo as a Kaggle dataset "
                            "or `git clone` it into /kaggle/working/Courtvision.")

REPO = find_repo()
sys.path.insert(0, str(REPO))
import courtvision
print("courtvision", courtvision.__version__, "from", REPO)

In [ ]:
import json
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

from courtvision import annotations as ann, config, data
from courtvision.court_mask import court_mask
from courtvision.geometry import corner_errors
from courtvision.homography import image_to_court, lane_court_points, project_court_to_image
from courtvision.lane import detect_segments, reconstruct_lane
from courtvision.propagation import chain_propagate
from courtvision.viz import BLUE, GREEN, RED, YELLOW, draw_infinite_line, draw_polygon, write_selector_html

OUT = config.KAGGLE_WORKING / "courtvision_phase3"
OUT.mkdir(parents=True, exist_ok=True)

SEQ_DIR, FRAME_PATHS = data.get_frame_paths()
H_IMG, W_IMG = cv2.imread(str(FRAME_PATHS[0])).shape[:2]
BEST_PT = data.find_checkpoint()
load_frame = lambda i: cv2.imread(str(FRAME_PATHS[i]))
print("Sequence:", SEQ_DIR, "| frames:", len(FRAME_PATHS), "| size:", (W_IMG, H_IMG))
print("Checkpoint:", BEST_PT)
print("OpenCV", cv2.__version__, "| NumPy", np.__version__)

## 2 · Stage 1 — Reproduce & audit the existing calibration

Nothing here changes the pipeline. It establishes the baseline we improve on.

In [ ]:
# 2.1 Frame indexing and annotation order
for i in (0, 100, 105, 115):
    print(f"FRAME_PATHS[{i:3d}] -> {FRAME_PATHS[i].name}   (tracking frame id {data.frame_id_of(FRAME_PATHS, i)})")

rows = []
for fid, a in ann.ANNOTATIONS.items():
    chk = ann.check_ft_order(a)
    rows.append({"frame": fid, "status": a.status, "ft_order": a.ft_order,
                 **{k: (round(v, 2) if isinstance(v, float) else v) for k, v in chk.items()}})
display(pd.DataFrame(rows))

MANUAL_LANE = {fid: ann.lane_corners(ann.ANNOTATIONS[fid]) for fid in (105, 115)}
for fid, lane in MANUAL_LANE.items():
    print(f"manual lane {fid}:", np.round(lane, 2).tolist())

In [ ]:
# 2.2 Reproduce the frame-115 lane reconstruction (original cells 34-38, identical parameters)
HANDOVER_AUTO_115 = np.array([[340.44, 540.66], [435.97, 437.65], [861.26, 482.80], [797.80, 592.70]])

f115 = load_frame(115)
segs = detect_segments(f115)  # full frame, as in the original
res115 = reconstruct_lane(segs, ann.seed_lines(ann.ANNOTATIONS[115]))
print("Hough segments:", 0 if segs is None else len(segs), "| status:", res115.status, "| support:", res115.counts)

if res115.corners is not None:
    per, mean, mx = corner_errors(res115.corners, MANUAL_LANE[115])
    print("auto corners:", np.round(res115.corners, 2).tolist())
    print(f"vs manual: {np.round(per, 2)}  mean {mean:.2f}  max {mx:.2f}   (handover: mean 4.10, max 6.86)")
    print("vs handover auto corners:", np.round(np.linalg.norm(res115.corners - HANDOVER_AUTO_115, axis=1), 2))
    ov = f115.copy()
    for hl in res115.lines.values():
        draw_infinite_line(ov, hl, YELLOW, 1)
    draw_polygon(ov, MANUAL_LANE[115], RED, 2)
    draw_polygon(ov, res115.corners, GREEN, 2)
    plt.figure(figsize=(16, 9)); plt.imshow(cv2.cvtColor(ov, cv2.COLOR_BGR2RGB)); plt.axis("off")
    plt.title("Frame 115 · green auto · red manual · yellow fitted lines"); plt.show()
    cv2.imwrite(str(OUT / "stage1_frame115_lane.png"), ov)
print("NOTE: line search is seeded by the manual clicks, so this measures agreement with the manual lines,"
      " not prior-free detection.")

In [ ]:
# 2.3 Which image points built the original "frame 100" reference homography?
CELL14_LANE = ann.lane_corners(ann.ANNOTATIONS[100])
REF_POINTS, REF_SOURCE = None, None
p_h = config.KAGGLE_WORKING / "courtvision_homography_frame100.json"
p_a = config.KAGGLE_WORKING / "courtvision_calibration_anchors.json"
if p_h.exists():
    REF_POINTS, REF_SOURCE = np.asarray(json.loads(p_h.read_text())["image_points"]), p_h.name
elif p_a.exists():
    H_a = np.asarray(json.loads(p_a.read_text())["anchors"]["100"]["homography"])
    REF_POINTS, REF_SOURCE = project_court_to_image(H_a), p_a.name + " (back-projected)"

if REF_POINTS is None:
    REF_POINTS, REF_SOURCE = CELL14_LANE, "ASSUMED: cell-14 lane (no saved artifact in /kaggle/working)"
else:
    print("distance to cell-14 lane:", np.round(np.linalg.norm(REF_POINTS - CELL14_LANE, axis=1), 1))
    print("distance to cell-15 lane:", np.round(np.linalg.norm(REF_POINTS - ann.FRAME75_LANE_CORNERS, axis=1), 1))
print("reference source:", REF_SOURCE)
print("reference points:", np.round(REF_POINTS, 2).tolist())
H_REF_100 = image_to_court(REF_POINTS)

In [ ]:
# 2.4 Chained propagation 100 -> 115: original full-frame ORB vs court-only ORB (diagnostic)
mask_fn = lambda f: court_mask(f)
records, chained = [], {}
for mode, fn in (("full_frame", None), ("court_only", mask_fn)):
    homs, recs = chain_propagate(load_frame, H_REF_100, 100, 116, mask_fn=fn, diag_mask_fn=mask_fn, seed=0)
    for r in recs:
        r["mode"] = mode
        if r["frame"] in MANUAL_LANE and r["frame"] in homs:
            chained[(mode, r["frame"])] = project_court_to_image(homs[r["frame"]])
            _, r["lane_err_mean_px"], r["lane_err_max_px"] = corner_errors(chained[(mode, r["frame"])],
                                                                           MANUAL_LANE[r["frame"]])
    records += recs

STAGE1 = pd.DataFrame(records).set_index(["mode", "frame"]).round(3)
display(STAGE1)
STAGE1.to_csv(OUT / "stage1_propagation_diagnostics.csv")
print("handover (full-frame chain): 105 mean 9.40 / max 13.41 ; 115 mean 62.92 / max 79.82")

fig, axes = plt.subplots(1, 2, figsize=(22, 7))
for ax, fid in zip(axes, (105, 115)):
    ov = load_frame(fid)
    draw_polygon(ov, MANUAL_LANE[fid], GREEN, 2)
    for mode, col in (("full_frame", RED), ("court_only", BLUE)):
        if (mode, fid) in chained:
            draw_polygon(ov, chained[(mode, fid)], col, 2)
    ax.imshow(cv2.cvtColor(ov, cv2.COLOR_BGR2RGB)); ax.axis("off")
    ax.set_title(f"Frame {fid} · green manual · red chained full-frame · blue chained court-only")
plt.tight_layout(); plt.savefig(OUT / "stage1_overlays_105_115.png", dpi=150, bbox_inches="tight"); plt.show()

## 3 · Annotation tool (optional)

Writes a standalone HTML page for clicking the 8 lane landmarks on any frame. Labels follow the
105/115 convention (P7 = FT-lower, P8 = FT-upper). Add results to `courtvision/annotations.py`
with `ft_order="lower_upper"`; never edit existing entries.

In [ ]:
ANNOTATE_FRAME = None  # e.g. 150
if ANNOTATE_FRAME is not None:
    print("Created:", write_selector_html(load_frame(ANNOTATE_FRAME), ANNOTATE_FRAME,
                                          OUT / f"selector_frame{ANNOTATE_FRAME}.html"))

## 4 · Phase 1–2 tracking (unchanged)

Off by default so calibration iterations don't spend GPU time. Set `RUN_TRACKING = True` or point
`EXISTING_TRACK_CSV` at a saved `tracks_pixel.csv`.

In [ ]:
from courtvision import tracking

RUN_TRACKING = False
EXISTING_TRACK_CSV = None  # e.g. "/kaggle/input/<dataset>/tracks_pixel.csv"

tracks = None
if EXISTING_TRACK_CSV:
    tracks = tracking.load_tracks(EXISTING_TRACK_CSV)
elif RUN_TRACKING:
    assert BEST_PT is not None, "Attach the Kaggle input containing best.pt"
    tracks = tracking.run_tracking(BEST_PT, FRAME_PATHS, tracking.write_tracker_yaml(OUT / "bytetrack_optimized.yaml"))
    tracks.to_csv(OUT / "tracks_pixel.csv", index=False)
if tracks is not None:
    tracks = tracking.add_footpoints(tracks)
    print("Observations:", len(tracks), "| unique track IDs:", tracks.track_id.nunique())
else:
    print("Tracking skipped.")

## 5 · Court mapping — gated

Player footpoints are mapped **only** for frames with an accepted per-frame calibration. None exist yet
(Stage 2+ produces them), so this section is intentionally inert. Output is lane-relative metres.

In [ ]:
from courtvision.homography import map_points

ACCEPTED_HOMOGRAPHIES = {}  # {FRAME_PATHS index: image->court H}; filled by validated calibration only

if tracks is not None and ACCEPTED_HOMOGRAPHIES:
    tracks["frame_index"] = tracks["frame"].map(lambda f: data.index_of(FRAME_PATHS, f))
    tracks[["lane_x_m", "lane_y_m"]] = np.nan
    for idx, H in ACCEPTED_HOMOGRAPHIES.items():
        sel = tracks.frame_index == idx
        tracks.loc[sel, ["lane_x_m", "lane_y_m"]] = map_points(H, tracks.loc[sel, ["foot_x_px", "foot_y_px"]].to_numpy())
    tracks.to_csv(OUT / "player_lane_coordinates.csv", index=False)
    print("Mapped frames:", len(ACCEPTED_HOMOGRAPHIES))
else:
    print("Mapping skipped: no validated per-frame calibration yet.")

## 6 · Export

In [ ]:
import shutil
meta = {"sequence": config.SEQUENCE_NAME, "frames": len(FRAME_PATHS), "imgsz": config.IMGSZ, "conf": config.CONF,
        "iou": config.IOU, "bytetrack": config.BYTETRACK_OVERRIDES, "reference_source_frame100": REF_SOURCE,
        "accepted_calibrated_frames": sorted(ACCEPTED_HOMOGRAPHIES),
        "limitations": ["No static whole-clip homography; per-frame calibration pending validation",
                        "Lane-relative coordinates; NCAA dimensions unverified for this court",
                        "Scoreboard occludes lane corners in many frames",
                        "Track IDs not guaranteed stable"]}
(OUT / "experiment_config.json").write_text(json.dumps(meta, indent=2, default=str))
archive = shutil.make_archive(str(config.KAGGLE_WORKING / "courtvision_phase3_artifacts"), "zip",
                              root_dir=OUT.parent, base_dir=OUT.name)
print("Archive:", archive)